In [1]:
import warnings

# === Step 0: Suppress Specific Warnings ===

# Suppress FutureWarnings from xgboost related to glibc
warnings.filterwarnings(
    "ignore",
    message="Your system has an old version of glibc.*",
    category=FutureWarning
)

# Optionally, suppress all FutureWarnings (if needed)
# warnings.filterwarnings("ignore", category=FutureWarning)

# Suppress all warnings (use with caution)
# warnings.filterwarnings("ignore")

# === Continue with Imports ===

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.linear_model import LogisticRegression
from joblib import Parallel, delayed

# === Step 1: Discretize all features to binary based on global median ===

def discretize_binary_global(X_df):
    """
    Discretize continuous features into binary based on the global median.

    Parameters:
    - X_df: DataFrame containing feature columns.

    Returns:
    - Discretized DataFrame with binary features.
    """
    X_discretized = X_df.copy()
    for column in X_discretized.columns:
        median = X_discretized[column].median()
        X_discretized[column] = (X_discretized[column] > median).astype(int)
    return X_discretized

# Load the dataset
data = pd.read_excel("class1_dataset.xlsx")  # Ensure the file path is correct

# Extract predictors (X) and outcome (Y)
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Apply global discretization
X_discretized = discretize_binary_global(X)

# Combine discretized features with the outcome
data_discretized = X_discretized.copy()
data_discretized['RRI'] = Y

# Initialize 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# === Step 2: Define Evaluation Function for (Alpha, k) ===

def evaluate_alpha_k(alpha, k, num_features):
    """
    Evaluate performance for a given alpha and feature subset size k.

    Parameters:
    - alpha: Regularization strength for Logistic LASSO.
    - k: Number of top features to select.
    - num_features: Total number of features.

    Returns:
    - Dictionary containing alpha, k, average AUC, std AUC, average Accuracy, std Accuracy.
    """
    # Initialize lists to store performance metrics across folds
    auc_scores = []
    accuracy_scores = []
    
    # Iterate over each fold
    for fold, (train_index, test_index) in enumerate(kf.split(data_discretized), 1):
        # Split data into training and testing sets
        train_data = data_discretized.iloc[train_index].copy()
        test_data = data_discretized.iloc[test_index].copy()

        # Separate predictors and outcome
        X_train = train_data.drop('RRI', axis=1)
        Y_train = train_data['RRI']
        X_test = test_data.drop('RRI', axis=1)
        Y_test = test_data['RRI']

        # Apply Logistic LASSO on training data
        try:
            lasso = LogisticRegression(
                penalty='l1',
                C=1/alpha if alpha != 0 else 1e10,  # Avoid division by zero
                solver='liblinear',
                max_iter=1000
            )
            lasso.fit(X_train, Y_train)
            coef = lasso.coef_[0]
            # Rank features by absolute coefficient values
            feature_ranking = np.argsort(np.abs(coef))[::-1]
            # Select top k features
            selected_feature_indexes = feature_ranking[:k]
            selected_feature_names = X_train.columns[selected_feature_indexes].tolist()

            if not selected_feature_names:
                # If no features are selected, skip this fold
                continue
        except Exception:
            # Skip this fold if feature selection fails
            continue

        # Prepare training and testing data with selected features
        train_selected = train_data[selected_feature_names].copy()
        train_selected['RRI'] = Y_train
        test_selected = test_data[selected_feature_names].copy()
        test_selected['RRI'] = Y_test

        # Learn the Bayesian Network structure using training data
        hc = HillClimbSearch(train_selected)
        try:
            best_model_structure = hc.estimate(scoring_method=BicScore(train_selected))
        except Exception:
            # Skip this fold if structure learning fails
            continue

        # Create and fit the Bayesian Network model
        model = BayesianNetwork(best_model_structure.edges())
        try:
            model.fit(train_selected, estimator=BayesianEstimator)
        except Exception:
            # Skip this fold if model fitting fails
            continue

        # Perform inference
        infer = VariableElimination(model)

        # Predict probabilities for the test set
        y_true = test_selected['RRI']
        y_pred_probs = []

        network_vars = set(model.nodes())

        for _, row in test_selected.iterrows():
            evidence = {col: row[col] for col in selected_feature_names if col in network_vars}
            try:
                result = infer.query(variables=['RRI'], evidence=evidence, show_progress=False)
                # Assuming 'RRI' has states [0, 1]
                y_pred_probs.append(result.values[1])  # Probability of class 1 (positive class)
            except Exception:
                y_pred_probs.append(0)  # Assign a default probability or handle appropriately

        # Convert probabilities to binary predictions
        y_pred = [1 if prob > 0.5 else 0 for prob in y_pred_probs]

        # Calculate performance metrics
        try:
            auc = roc_auc_score(y_true, y_pred_probs)
        except ValueError:
            auc = 0.5  # Assign a default AUC if only one class is present
        accuracy = accuracy_score(y_true, y_pred)

        # Append metrics
        auc_scores.append(auc)
        accuracy_scores.append(accuracy)

    # After all folds, compute average and standard deviation of metrics
    if auc_scores and accuracy_scores:
        avg_auc = np.mean(auc_scores)
        std_auc = np.std(auc_scores)
        avg_accuracy = np.mean(accuracy_scores)
        std_accuracy = np.std(accuracy_scores)
    else:
        # If no valid results were obtained
        avg_auc = 0
        std_auc = 0
        avg_accuracy = 0
        std_accuracy = 0

    # Return the performance metrics without printing intermediate results
    return {
        'alpha': alpha,
        'k': k,
        'avg_auc': avg_auc,
        'std_auc': std_auc,
        'avg_accuracy': avg_accuracy,
        'std_accuracy': std_accuracy
    }

# === Step 3: Evaluate Different (Alpha, k) Combinations in Parallel ===

# Define the range of alpha values (logspace from 0.00001 to 0.1)
alphas = np.logspace(-5, -1, 40)

# Number of features
num_features = X_discretized.shape[1]

# Generate all possible (alpha, k) pairs
alpha_k_pairs = [(alpha, k, num_features) for alpha in alphas for k in range(1, num_features + 1)]

# Function to evaluate a single (alpha, k) pair
def evaluate_pair(pair):
    alpha, k, num_features = pair
    return evaluate_alpha_k(alpha, k, num_features)

# Set up parallel processing for different (alpha, k) pairs
# Using n_jobs=9 as per your original setting
results_alpha_k = Parallel(n_jobs=9, verbose=0)(
    delayed(evaluate_pair)(pair) for pair in alpha_k_pairs
)

# === Step 4: Identify and Store the Best Performing Feature Subset ===

# Convert the results to a DataFrame
results_df = pd.DataFrame(results_alpha_k)

# Remove any rows with zero AUC (optional, based on your data)
results_df = results_df[results_df['avg_auc'] > 0]

# Identify the (alpha, k) pair with the highest average AUC
if not results_df.empty:
    best_result = results_df.loc[results_df['avg_auc'].idxmax()]
    best_alpha = best_result['alpha']
    best_k = int(best_result['k'])  # Convert to integer here
    best_auc = best_result['avg_auc']
    best_auc_std = best_result['std_auc']
    best_accuracy = best_result['avg_accuracy']
    best_accuracy_std = best_result['std_accuracy']
else:
    best_alpha = None
    best_k = None
    best_auc = None
    best_auc_std = None
    best_accuracy = None
    best_accuracy_std = None

# === Step 5: Fit Logistic LASSO on Entire Dataset with Best Alpha and Select Top k Features ===

if best_alpha is not None and best_k is not None:
    try:
        lasso = LogisticRegression(
            penalty='l1',
            C=1/best_alpha if best_alpha != 0 else 1e10,  # Avoid division by zero
            solver='liblinear',
            max_iter=1000
        )
        lasso.fit(X_discretized, Y)
        coef = lasso.coef_[0]
        # Rank features by absolute coefficient values
        feature_ranking = np.argsort(np.abs(coef))[::-1]
        # Select top k features
        best_selected_feature_indexes = feature_ranking[:best_k].tolist()
    except Exception as e:
        print(f"    Logistic LASSO failed when fitting on entire dataset for best alpha={best_alpha:.5f} and k={best_k}: {e}")
        best_selected_feature_indexes = []
else:
    best_selected_feature_indexes = []


# === Step 6: Display the Final Results ===

print("\n=== Feature Selection Results ===")
if best_alpha is not None and best_k is not None:
    print(f"Best alpha: {best_alpha:.5f}")
    print(f"Best number of features (k): {best_k}")
    print(f"Selected feature indexes: {best_selected_feature_indexes}")
    print(f"Best Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
    print(f"Best Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
else:
    print("No valid results were obtained.")

/home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
  0%|          | 1/1000000 [00:00<1:54:43, 145.27it/s]/home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the

    Logistic LASSO failed when fitting on entire dataset for best alpha=0.00001 and k=19.0: slice indices must be integers or None or have an __index__ method

=== Feature Selection Results ===
Best alpha: 0.00001
Best number of features (k): 19.0
Selected feature indexes: []
Best Average AUC: 0.6494 ± 0.0389
Best Average Accuracy: 0.9088 ± 0.0102


In [2]:
# === Step 4: Identify and Store the Best Performing Feature Subset ===

# Convert the results to a DataFrame
results_df = pd.DataFrame(results_alpha_k)

# Remove any rows with zero AUC (optional, based on your data)
results_df = results_df[results_df['avg_auc'] > 0]

# Identify the (alpha, k) pair with the highest average AUC
if not results_df.empty:
    best_result = results_df.loc[results_df['avg_auc'].idxmax()]
    best_alpha = best_result['alpha']
    best_k = int(best_result['k'])  # Convert to integer here
    best_auc = best_result['avg_auc']
    best_auc_std = best_result['std_auc']
    best_accuracy = best_result['avg_accuracy']
    best_accuracy_std = best_result['std_accuracy']
else:
    best_alpha = None
    best_k = None
    best_auc = None
    best_auc_std = None
    best_accuracy = None
    best_accuracy_std = None

# === Step 5: Fit Logistic LASSO on Entire Dataset with Best Alpha and Select Top k Features ===

if best_alpha is not None and best_k is not None:
    try:
        lasso = LogisticRegression(
            penalty='l1',
            C=1/best_alpha if best_alpha != 0 else 1e10,  # Avoid division by zero
            solver='liblinear',
            max_iter=1000
        )
        lasso.fit(X_discretized, Y)
        coef = lasso.coef_[0]
        # Rank features by absolute coefficient values
        feature_ranking = np.argsort(np.abs(coef))[::-1]
        # Select top k features
        best_selected_feature_indexes = feature_ranking[:best_k].tolist()
    except Exception as e:
        print(f"    Logistic LASSO failed when fitting on entire dataset for best alpha={best_alpha:.5f} and k={best_k}: {e}")
        best_selected_feature_indexes = []
else:
    best_selected_feature_indexes = []


# === Step 6: Display the Final Results ===

print("\n=== Feature Selection Results ===")
if best_alpha is not None and best_k is not None:
    print(f"Best alpha: {best_alpha:.5f}")
    print(f"Best number of features (k): {best_k}")
    print(f"Selected feature indexes: {best_selected_feature_indexes}")
    print(f"Best Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
    print(f"Best Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
else:
    print("No valid results were obtained.")


=== Feature Selection Results ===
Best alpha: 0.00001
Best number of features (k): 19
Selected feature indexes: [24, 30, 28, 9, 3, 12, 13, 8, 2, 10, 7, 6, 27, 25, 36, 20, 33, 19, 15]
Best Average AUC: 0.6494 ± 0.0389
Best Average Accuracy: 0.9088 ± 0.0102
